# 07 - Operational-resilient optimization model

The main model. A mixed-integer linear program that jointly selects active hubs, sizes
the drone fleet at each hub and assigns standardized packages, subject to
return-to-origin endurance, response-time feasibility, mission-cycle capacity and
backup accessibility for high-priority cells. Solved lexicographically: hubs, then
fleet, then response time.

**Requires a solver.** 16 fixed-p diagnostics, 2 baseline runs and 10 sensitivity
scenarios. **Output:** `data/model/operational_resilient_solutions_v4/`.

## 0. Modeling assumptions and interpretation

### Standardized medical shipment

Each dispatch transports **one standardized emergency medical-supply package**. The package is assumed to be within the payload capability of the standardized drone platform. Payload weight is therefore not a decision variable in this experiment.

### Demand interpretation

The OSM-informed demand weights from notebook 03 are **relative spatial demand proxies**, not observed medical incidents or package requests.

For the capacity experiment, the model specifies a system-level workload in standardized packages per hour and allocates those packages spatially in proportion to the proxy-demand weights.

The baseline workload is:

$$
Q = 30 \text{ packages/hour}.
$$

Sensitivity scenarios use:

$$
Q \in \{15,30,45\}.
$$

These values are research scenarios and must not be reported as observed Hajj emergency-demand rates.

### Drone fleet

Each active hub can host at most:

$$
N^{\max}=3
$$

drones.

The number of drones at each selected hub is an integer decision variable.

### Return-to-origin operation

Every mission follows:

$$
\text{hub} \rightarrow \text{demand point} \rightarrow \text{same hub}.
$$

Thus, if $D_{ij}$ is the one-way straight-line distance from hub $j$ to demand cell $i$, the modeled mission distance is:

$$
L^{RT}_{ij}=2D_{ij}.
$$

### Safe battery endurance

The baseline safe round-trip mission-distance budget is:

$$
L^{safe}=20 \text{ km}.
$$

Sensitivity scenarios use:

$$
L^{safe}\in\{10,15,20\}\text{ km}.
$$

This is a conservative research assumption informed by commercial drone capability. It is not treated as a manufacturer-certified Hajj operating range.

### Response time

The baseline emergency-response target is:

$$
T^{\max}=5\text{ min}.
$$

Sensitivity scenarios use:

$$
T^{\max}\in\{5,7,10\}\text{ min}.
$$

Operational cruise speed is assumed to be:

$$
v=15\text{ m/s}=0.9\text{ km/min}.
$$

A fixed:

$$
T_0=1\text{ min}
$$

allowance represents dispatch, ascent/descent, and handover activities.

Therefore:

$$
T^{resp}_{ij}=T_0+\frac{D_{ij}}{v}.
$$

### Turnaround and fleet capacity

After returning to the hub, a drone requires turnaround time for recharge/battery handling and re-equipment.

Baseline:

$$
T^{turn}=30\text{ min}.
$$

Sensitivity:

$$
T^{turn}\in\{15,30,45\}\text{ min}.
$$

The complete mission-cycle workload is:

$$
T^{cycle}_{ij}=\frac{2D_{ij}}{v}+T^{turn}.
$$

### Resilience

High-priority demand cells are defined as the top 20% of cells by OSM-informed proxy-demand weight.

In the resilient baseline, each critical cell must be operationally reachable from at least:

$$
r=2
$$

independent active hubs.

A nonredundant comparison with $r=1$ is retained to quantify the resilience premium.

### Evidence basis

The 5-minute emergency-response target, a maximum of three drones per base, and a 30-minute recharge/re-equipment assumption have precedent in published AED-drone network research. Commercial DJI Matrice 350 RTK specifications provide a physical reference point for payload, speed, endurance, and charging capability. Battery- and capacity-constrained drone facility-location modeling has precedent in the operations-research literature, while backup coverage has precedent in medical-drone location planning.

The 20 km safe mission budget, 15 m/s cruise speed, one-minute fixed overhead, top-20% critical threshold, and package-per-hour levels remain explicit scenario assumptions and will be reported as such.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import pyomo.environ as pyo

PROJECT_ROOT = Path("hajj_drone_project")
MODEL_DIR = PROJECT_ROOT / "data" / "model"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
SOLUTION_DIR = MODEL_DIR / "operational_resilient_solutions_v4"

for folder in [REPORT_DIR, SOLUTION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Operational assumptions
MAX_DRONES_PER_HUB = 3
CRUISE_SPEED_MPS = 15.0
CRUISE_SPEED_KM_PER_MIN = CRUISE_SPEED_MPS * 60 / 1000
FIXED_RESPONSE_OVERHEAD_MIN = 1.0
CRITICAL_SHARE = 0.20

# Baseline scenario
BASELINE_PACKAGES_PER_HOUR = 30
BASELINE_SAFE_ROUNDTRIP_KM = 20.0
BASELINE_RESPONSE_TARGET_MIN = 5.0
BASELINE_TURNAROUND_MIN = 30.0
BASELINE_REDUNDANCY = 2

# Fixed-p diagnostic range. This is NOT the main sensitivity design.
P_DIAGNOSTIC = list(range(5, 13))

solver = pyo.SolverFactory("highs")

if not solver.available(exception_flag=False):
    raise RuntimeError("Install with: %pip install -U pyomo highspy")

print("Solver: HiGHS through Pyomo")
print("Output directory:", SOLUTION_DIR.resolve())

Solver: HiGHS through Pyomo
Output directory: C:\Users\firas.hindawi.CCM\Desktop\papers\Drone-Based Data Driven Medical Supply Distribution\hajj-drone-logistics\notebooks\hajj_drone_project\data\model\operational_resilient_solutions_v4


## 1. Load the frozen spatial model inputs

The operational experiment uses exactly the same baseline demand points, candidate hubs, and demand-to-hub distance matrix used by notebook 05.

This preserves comparability between the classical and operational experiments.

The expected model dimensions are:

- 556 demand cells;
- 59 candidate medical-drone hubs;
- a $556 \times 59$ straight-line distance matrix.

In [ ]:
demand_file = MODEL_DIR / "baseline_500m_demand_points_with_nearest_hub_v1.csv"
hub_file = MODEL_DIR / "candidate_drone_hubs_v1.csv"
distance_file = MODEL_DIR / "demand_to_candidate_hub_air_distance_km_v1.csv"

for f in [demand_file, hub_file, distance_file]:
    if not f.exists():
        raise FileNotFoundError(f"Missing {f}. Run notebook 04 first.")

demand = pd.read_csv(demand_file)
hubs = pd.read_csv(hub_file)
distance_df = pd.read_csv(distance_file, index_col=0)

for df in [demand, hubs]:
    bad = [c for c in df.columns if str(c).lower().startswith("unnamed:")]
    if bad:
        df.drop(columns=bad, inplace=True)

# Align demand rows exactly with the distance matrix.
demand = (
    demand.set_index("demand_point_id")
    .loc[distance_df.index]
    .reset_index()
)

matrix_hub_ids = list(distance_df.columns)

if "hub_id" not in hubs.columns:
    hubs.insert(0, "hub_id", matrix_hub_ids)
elif hubs["hub_id"].astype(str).tolist() != matrix_hub_ids:
    hubs = hubs.set_index("hub_id").loc[matrix_hub_ids].reset_index()

D = distance_df.to_numpy(float)
ROUNDTRIP_KM = 2 * D

I = list(range(len(demand)))
J = list(range(len(hubs)))

assert D.shape == (556, 59)

print("Demand cells:", len(I))
print("Candidate hubs:", len(J))
print("Distance matrix:", D.shape)

Demand cells: 556
Candidate hubs: 59
Distance matrix: (556, 59)


## 2. Construct scenario package demand

Let $w_i$ denote the OSM-informed proxy-demand weight of demand cell $i$.

For a scenario with $Q$ standardized packages per hour, continuous proportional demand is:

$$
\tilde q_i =
Q
\frac{w_i}{\sum_k w_k}.
$$

Because a standardized shipment is indivisible, the largest-remainder method converts these proportional values into integer package counts $q_i$ satisfying:

$$
\sum_i q_i=Q.
$$

This transformation uses the OSM proxy only to determine the **relative spatial allocation** of the scenario workload.

In [ ]:
proxy_w = demand["proxy_demand_weight"].to_numpy(float)

def allocate_integer_demand(total_packages):
    raw = total_packages * proxy_w / proxy_w.sum()
    q = np.floor(raw).astype(int)

    remainder = int(total_packages - q.sum())

    if remainder > 0:
        order = np.argsort(-(raw - q))
        q[order[:remainder]] += 1

    assert q.sum() == total_packages
    return q

for Q in [15, 30, 45]:
    q = allocate_integer_demand(Q)
    print(
        f"{Q} packages/hour:",
        int((q > 0).sum()),
        "active demand cells"
    )

15 packages/hour: 15 active demand cells
30 packages/hour: 30 active demand cells
45 packages/hour: 45 active demand cells


## 3. Define high-priority cells for resilient preparedness

Critical cells are defined using the upper 20% of the proxy-demand distribution.

Let $I^C$ denote this set.

The resilient model requires each critical cell to have at least two independently feasible active hubs. This is a **preparedness requirement**, so it applies even when a particular critical cell does not receive a package in the specific hourly demand realization.

The 20% threshold is a transparent scenario choice rather than a clinical cutoff.

In [ ]:
critical_threshold = demand["proxy_demand_weight"].quantile(
    1 - CRITICAL_SHARE
)

critical_mask = (
    demand["proxy_demand_weight"].to_numpy(float)
    >= critical_threshold
)

critical_indices = np.where(critical_mask)[0].tolist()

print("Critical threshold:", critical_threshold)
print("Critical cells:", len(critical_indices))
print("Critical share:", len(critical_indices) / len(demand))

display(
    demand.loc[
        critical_mask,
        ["demand_point_id", "hajj_zone", "proxy_demand_weight"]
    ]
    .sort_values("proxy_demand_weight", ascending=False)
    .head(15)
)

Critical threshold: 1.2736156351791532
Critical cells: 112
Critical share: 0.2014388489208633


,demand_point_id,hajj_zone,proxy_demand_weight
111,D_500_Haram_Makkah_0112,Haram_Makkah,5.000000
257,D_500_Mina_0050,Mina,4.940065
280,D_500_Mina_0073,Mina,4.788925
235,D_500_Mina_0028,Mina,4.385016
269,D_500_Mina_0062,Mina,4.267752
93,D_500_Haram_Makkah_0094,Haram_Makkah,4.197394
281,D_500_Mina_0074,Mina,3.850814
95,D_500_Haram_Makkah_0096,Haram_Makkah,3.777850
270,D_500_Mina_0063,Mina,3.749186
245,D_500_Mina_0038,Mina,3.738762


## 4. Operational feasibility and mission-cycle matrices

Battery feasibility requires:

$$
2D_{ij}\le L^{safe}.
$$

Response-time feasibility requires:

$$
T_0+\frac{D_{ij}}{v}\le T^{\max}.
$$

A hub-demand pair is operationally feasible only when **both** conditions hold.

For capacity, one standardized package assigned from hub $j$ to cell $i$ consumes:

$$
T^{cycle}_{ij}
=
\frac{2D_{ij}}{v}
+
T^{turn}
$$

drone-minutes.

Because turnaround is sensitivity-tested, the cycle-time matrix is recalculated for each scenario.

In [ ]:
RESPONSE_MIN = (
    FIXED_RESPONSE_OVERHEAD_MIN
    + D / CRUISE_SPEED_KM_PER_MIN
)

def feasibility_matrix(
    safe_roundtrip_km,
    response_target_min
):
    return (
        (ROUNDTRIP_KM <= safe_roundtrip_km)
        &
        (RESPONSE_MIN <= response_target_min)
    )

def cycle_matrix(turnaround_min):
    return (
        ROUNDTRIP_KM / CRUISE_SPEED_KM_PER_MIN
        + turnaround_min
    )

## 5. Baseline structural diagnostics

Before solving the MILP, the notebook diagnoses whether battery endurance, response time, or redundancy is the restrictive requirement.

This is useful because MILP infeasibility alone does not reveal which operational assumption is responsible.

In [ ]:
q = allocate_integer_demand(BASELINE_PACKAGES_PER_HOUR)

battery_feasible = (
    ROUNDTRIP_KM
    <= BASELINE_SAFE_ROUNDTRIP_KM
)

response_feasible = (
    RESPONSE_MIN
    <= BASELINE_RESPONSE_TARGET_MIN
)

combined_feasible = (
    battery_feasible
    &
    response_feasible
)

combined_counts = combined_feasible.sum(axis=1)

print("=" * 68)
print("BASELINE STRUCTURAL DIAGNOSTICS")
print("=" * 68)

print(
    "Cells with zero battery-feasible hubs:",
    int((battery_feasible.sum(axis=1) == 0).sum())
)

print(
    "Cells with zero response-feasible hubs:",
    int((response_feasible.sum(axis=1) == 0).sum())
)

print(
    "Active-demand cells with zero operationally feasible hubs:",
    int((combined_counts[q > 0] == 0).sum())
)

print(
    "Critical cells with fewer than two feasible hubs:",
    int((combined_counts[np.array(critical_indices)] < 2).sum())
)

print(
    "Hub-demand pairs allowed by battery but rejected by response time:",
    int((battery_feasible & ~response_feasible).sum())
)

print(
    "Hub-demand pairs allowed by response time but rejected by battery:",
    int((response_feasible & ~battery_feasible).sum())
)

BASELINE STRUCTURAL DIAGNOSTICS
Cells with zero battery-feasible hubs: 0
Cells with zero response-feasible hubs: 2
Active-demand cells with zero operationally feasible hubs: 0
Critical cells with fewer than two feasible hubs: 0
Hub-demand pairs allowed by battery but rejected by response time: 13183
Hub-demand pairs allowed by response time but rejected by battery: 0


## 6. Aggregate fleet-capacity lower bound

A useful lower bound can be calculated before optimization.

Each active demand cell is allowed to use its individually best operationally feasible hub. This ignores the requirement that all assignments must be supported by one common selected hub network, so it is an optimistic lower bound.

The resulting minimum workload is:

$$
W^{LB}
=
\sum_i q_i
\min_{j\in F_i}
T^{cycle}_{ij}.
$$

The corresponding lower bound on drones is:

$$
N^{LB}
=
\left\lceil
\frac{W^{LB}}{60}
\right\rceil.
$$

With at most three drones per hub, a capacity-only hub lower bound is:

$$
P^{LB}
=
\left\lceil
\frac{N^{LB}}{3}
\right\rceil.
$$

The gap between this aggregate lower bound and the exact MILP solution measures the additional infrastructure required because capacity must be placed at geographically compatible hubs.

In [ ]:
q = allocate_integer_demand(BASELINE_PACKAGES_PER_HOUR)
feasible = feasibility_matrix(
    BASELINE_SAFE_ROUNDTRIP_KM,
    BASELINE_RESPONSE_TARGET_MIN
)
cycle = cycle_matrix(BASELINE_TURNAROUND_MIN)

minimum_workload = 0.0

for i in np.where(q > 0)[0]:
    js = np.where(feasible[i])[0]

    if len(js) > 0:
        minimum_workload += (
            q[i]
            * cycle[i, js].min()
        )

minimum_drones_lb = int(
    np.ceil(minimum_workload / 60)
)

minimum_hubs_lb = int(
    np.ceil(
        minimum_drones_lb
        / MAX_DRONES_PER_HUB
    )
)

print("Best-case workload:", round(minimum_workload, 2), "drone-minutes/hour")
print("Fleet lower bound:", minimum_drones_lb)
print("Capacity-only hub lower bound:", minimum_hubs_lb)

Best-case workload: 942.16 drone-minutes/hour
Fleet lower bound: 16
Capacity-only hub lower bound: 6


## 7. Fixed-hub-count operational model

This model is used as a **diagnostic experiment**, not as the primary final formulation.

For a specified number of active hubs $p$, the model determines the smallest feasible drone fleet and then the fastest package allocation at that fleet size.

### Decision variables

$y_j=1$ if candidate hub $j$ is active.

$n_j$ is the integer number of drones stationed at hub $j$.

$x_{ij}$ is the integer number of standardized packages per hour assigned from hub $j$ to demand cell $i$.

### Demand satisfaction

$$
\sum_j x_{ij}=q_i
\qquad \forall i.
$$

### Hub activation

$$
x_{ij}\le q_i y_j.
$$

### Fixed number of active hubs

$$
\sum_j y_j=p.
$$

### Fleet-to-hub linkage

$$
y_j\le n_j\le N^{\max}y_j.
$$

Thus, an active hub has at least one drone and at most three.

### Operational feasibility

If a hub-demand pair violates either battery endurance or response time, then:

$$
x_{ij}=0.
$$

### Fleet-derived hub capacity

Each drone provides 60 drone-minutes of hourly operational capacity:

$$
\sum_i
T^{cycle}_{ij}x_{ij}
\le
60n_j.
$$

### Redundant preparedness

For every critical cell $i\in I^C$:

$$
\sum_{j\in F_i}y_j\ge r,
$$

where $F_i$ contains hubs satisfying both battery and response-time feasibility.

### Lexicographic objective

For fixed $p$:

1. minimize total drones;
2. fix that minimum fleet and minimize total package response time.

This avoids combining unlike units in an arbitrary weighted objective.

In [ ]:
def _load_solution(model, result):
    try:
        model.solutions.load_from(result)
    except Exception:
        try:
            result.solution_loader.load_vars()
        except Exception:
            pass


def solve_fixed_p(
    p,
    total_packages=30,
    safe_roundtrip_km=20.0,
    response_target_min=5.0,
    turnaround_min=30.0,
    redundancy=2
):

    q = allocate_integer_demand(total_packages)

    feasible = feasibility_matrix(
        safe_roundtrip_km,
        response_target_min
    )

    cycle = cycle_matrix(
        turnaround_min
    )

    active = np.where(q > 0)[0]

    if any(
        not feasible[i].any()
        for i in active
    ):
        return {
            "p": p,
            "termination": "infeasible_reachability"
        }, None

    if redundancy > 1:

        bad = [
            i for i in critical_indices
            if feasible[i].sum() < redundancy
        ]

        if bad:
            return {
                "p": p,
                "termination": "infeasible_redundancy_geometry",
                "critical_cells_without_backup": len(bad)
            }, None

    m = pyo.ConcreteModel()

    m.I = pyo.RangeSet(0, len(I) - 1)
    m.J = pyo.RangeSet(0, len(J) - 1)

    m.y = pyo.Var(
        m.J,
        domain=pyo.Binary
    )

    m.n = pyo.Var(
        m.J,
        domain=pyo.NonNegativeIntegers,
        bounds=(0, MAX_DRONES_PER_HUB)
    )

    m.x = pyo.Var(
        m.I,
        m.J,
        domain=pyo.NonNegativeIntegers
    )

    m.demand = pyo.Constraint(
        m.I,
        rule=lambda m, i:
        sum(m.x[i, j] for j in m.J)
        == int(q[i])
    )

    m.link = pyo.Constraint(
        m.I,
        m.J,
        rule=lambda m, i, j:
        m.x[i, j]
        <= int(q[i]) * m.y[j]
    )

    m.hub_count = pyo.Constraint(
        expr=sum(m.y[j] for j in m.J)
        == p
    )

    m.fleet_lower = pyo.Constraint(
        m.J,
        rule=lambda m, j:
        m.n[j] >= m.y[j]
    )

    m.fleet_upper = pyo.Constraint(
        m.J,
        rule=lambda m, j:
        m.n[j]
        <= MAX_DRONES_PER_HUB * m.y[j]
    )

    m.forbidden = pyo.ConstraintList()

    for i in I:
        for j in J:
            if not feasible[i, j]:
                m.forbidden.add(
                    m.x[i, j] == 0
                )

    m.capacity = pyo.Constraint(
        m.J,
        rule=lambda m, j:
        sum(
            float(cycle[i, j]) * m.x[i, j]
            for i in I
        )
        <= 60 * m.n[j]
    )

    m.backup = pyo.ConstraintList()

    if redundancy > 1:

        for i in critical_indices:

            feasible_hubs = [
                j for j in J
                if feasible[i, j]
            ]

            m.backup.add(
                sum(
                    m.y[j]
                    for j in feasible_hubs
                )
                >= redundancy
            )

    # Stage 1: minimum fleet
    m.objective_fleet = pyo.Objective(
        expr=sum(
            m.n[j]
            for j in m.J
        )
    )

    t = time.perf_counter()

    r1 = solver.solve(
        m,
        load_solutions=False
    )

    solve_time = (
        time.perf_counter() - t
    )

    term1 = str(
        r1.solver.termination_condition
    )

    if term1.lower() != "optimal":
        return {
            "p": p,
            "termination": term1,
            "solve_time_s": solve_time
        }, None

    _load_solution(m, r1)

    minimum_drones = int(
        round(
            sum(
                pyo.value(m.n[j])
                for j in J
            )
        )
    )

    # Stage 2: minimum response time
    m.fleet_fix = pyo.Constraint(
        expr=sum(
            m.n[j]
            for j in m.J
        )
        == minimum_drones
    )

    m.objective_fleet.deactivate()

    m.objective_response = pyo.Objective(
        expr=sum(
            float(RESPONSE_MIN[i, j])
            * m.x[i, j]
            for i in I
            for j in J
        )
    )

    t = time.perf_counter()

    r2 = solver.solve(
        m,
        load_solutions=False
    )

    solve_time += (
        time.perf_counter() - t
    )

    term2 = str(
        r2.solver.termination_condition
    )

    if term2.lower() != "optimal":
        return {
            "p": p,
            "termination": term2,
            "minimum_drones": minimum_drones,
            "solve_time_s": solve_time
        }, None

    _load_solution(m, r2)

    selected = [
        j for j in J
        if pyo.value(m.y[j]) > 0.5
    ]

    drones = {
        j: int(round(pyo.value(m.n[j])))
        for j in selected
    }

    flows = {
        (i, j): int(round(pyo.value(m.x[i, j])))
        for i in I
        for j in J
        if (
            pyo.value(m.x[i, j]) is not None
            and pyo.value(m.x[i, j]) > 0.5
        )
    }

    responses = []
    distances = []

    for (i, j), units in flows.items():
        responses.extend(
            [RESPONSE_MIN[i, j]] * units
        )
        distances.extend(
            [D[i, j]] * units
        )

    metrics = {
        "p": p,
        "packages_per_hour": total_packages,
        "safe_roundtrip_km": safe_roundtrip_km,
        "response_target_min": response_target_min,
        "turnaround_min": turnaround_min,
        "redundancy": redundancy,
        "minimum_drones": minimum_drones,
        "mean_response_min": float(np.mean(responses)),
        "max_response_min": float(np.max(responses)),
        "mean_oneway_distance_km": float(np.mean(distances)),
        "max_oneway_distance_km": float(np.max(distances)),
        "termination": term2,
        "solve_time_s": solve_time,
        "selected_hubs": ";".join(
            hubs.iloc[selected]["hub_id"].astype(str)
        ),
        "drone_allocation": ";".join(
            f'{hubs.iloc[j]["hub_id"]}:{drones[j]}'
            for j in selected
        )
    }

    solution = {
        "selected": selected,
        "drones": drones,
        "flows": flows,
        "q": q,
        "feasible": feasible,
        "cycle": cycle
    }

    return metrics, solution

## 8. Diagnostic experiment A: operational feasibility threshold

The aggregate capacity lower bound does not account for the fact that fleet capacity must be located at a common set of geographically feasible hubs.

The fixed-$p$ model is therefore solved for:

$$
p\in\{5,6,7,8,9,10,11,12\}.
$$

This diagnostic identifies the exact operational feasibility threshold under the baseline assumptions.

It is **not** the main sensitivity experiment. Its purpose is to explain the difference between aggregate theoretical capacity and spatially feasible network capacity.

In [ ]:
fixed_p_baseline_results = []
fixed_p_baseline_solutions = {}

for p in P_DIAGNOSTIC:

    metrics, solution = solve_fixed_p(
        p=p,
        total_packages=BASELINE_PACKAGES_PER_HOUR,
        safe_roundtrip_km=BASELINE_SAFE_ROUNDTRIP_KM,
        response_target_min=BASELINE_RESPONSE_TARGET_MIN,
        turnaround_min=BASELINE_TURNAROUND_MIN,
        redundancy=BASELINE_REDUNDANCY
    )

    fixed_p_baseline_results.append(
        metrics
    )

    fixed_p_baseline_solutions[p] = (
        solution
    )

    print(
        "p =", p,
        "|", metrics["termination"],
        "| drones =",
        metrics.get("minimum_drones")
    )

fixed_p_baseline_df = pd.DataFrame(
    fixed_p_baseline_results
)

display(fixed_p_baseline_df)

## 9. Diagnostic experiment B: resilience premium

To isolate the effect of backup coverage, the fixed-$p$ experiment is repeated for:

$$
r=1
$$

and:

$$
r=2.
$$

All other baseline assumptions are held constant.

The difference in the minimum feasible hub and fleet requirements quantifies the **resilience premium**.

This experiment distinguishes additional spatial infrastructure required for backup preparedness from infrastructure required merely to serve the realized package workload.

In [ ]:
resilience_results = []
resilience_solutions = {}

for redundancy in [1, 2]:

    for p in P_DIAGNOSTIC:

        metrics, solution = solve_fixed_p(
            p=p,
            total_packages=BASELINE_PACKAGES_PER_HOUR,
            safe_roundtrip_km=BASELINE_SAFE_ROUNDTRIP_KM,
            response_target_min=BASELINE_RESPONSE_TARGET_MIN,
            turnaround_min=BASELINE_TURNAROUND_MIN,
            redundancy=redundancy
        )

        resilience_results.append(
            metrics
        )

        resilience_solutions[
            (redundancy, p)
        ] = solution

        print(
            "r =", redundancy,
            "| p =", p,
            "|", metrics["termination"],
            "| drones =",
            metrics.get("minimum_drones")
        )

resilience_df = pd.DataFrame(
    resilience_results
)

display(resilience_df)

r = 1 | p = 5 | infeasible | drones = None
r = 1 | p = 6 | infeasible | drones = None
r = 1 | p = 7 | optimal | drones = 19
r = 1 | p = 8 | optimal | drones = 19
r = 1 | p = 9 | optimal | drones = 20
r = 1 | p = 10 | optimal | drones = 20
r = 1 | p = 11 | optimal | drones = 21
r = 1 | p = 12 | optimal | drones = 21
r = 2 | p = 5 | infeasible | drones = None
r = 2 | p = 6 | infeasible | drones = None
r = 2 | p = 7 | infeasible | drones = None
r = 2 | p = 8 | infeasible | drones = None
r = 2 | p = 9 | infeasible | drones = None
r = 2 | p = 10 | optimal | drones = 21
r = 2 | p = 11 | optimal | drones = 22
r = 2 | p = 12 | optimal | drones = 22


,p,termination,solve_time_s,packages_per_hour,safe_roundtrip_km,response_target_min,turnaround_min,redundancy,minimum_drones,mean_response_min,max_response_min,mean_oneway_distance_km,max_oneway_distance_km,selected_hubs,drone_allocation
0,5,infeasible,3.727034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6,infeasible,10.548367,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7,optimal,13.057498,30.0,20.0,5.0,30.0,1.0,19.0,1.837398,3.028929,0.753658,1.826036,HUB_006;HUB_014;HUB_016;HUB_040;HUB_041;HUB_04...,HUB_006:3;HUB_014:2;HUB_016:3;HUB_040:3;HUB_04...
3,8,optimal,8.745018,30.0,20.0,5.0,30.0,1.0,19.0,1.780848,3.172634,0.702763,1.955371,HUB_006;HUB_014;HUB_016;HUB_040;HUB_041;HUB_04...,HUB_006:2;HUB_014:1;HUB_016:3;HUB_040:3;HUB_04...
4,9,optimal,9.782417,30.0,20.0,5.0,30.0,1.0,20.0,1.769958,3.172634,0.692963,1.955371,HUB_006;HUB_014;HUB_016;HUB_040;HUB_041;HUB_04...,HUB_006:2;HUB_014:1;HUB_016:3;HUB_040:3;HUB_04...
5,10,optimal,11.540824,30.0,20.0,5.0,30.0,1.0,20.0,1.778546,3.172634,0.700691,1.955371,HUB_006;HUB_012;HUB_014;HUB_016;HUB_040;HUB_04...,HUB_006:2;HUB_012:1;HUB_014:1;HUB_016:3;HUB_04...
6,11,optimal,6.615230,30.0,20.0,5.0,30.0,1.0,21.0,1.769834,3.172634,0.692851,1.955371,HUB_006;HUB_012;HUB_014;HUB_015;HUB_016;HUB_04...,HUB_006:2;HUB_012:1;HUB_014:1;HUB_015:1;HUB_01...
7,12,optimal,12.711768,30.0,20.0,5.0,30.0,1.0,21.0,1.798699,3.172634,0.718829,1.955371,HUB_006;HUB_012;HUB_014;HUB_015;HUB_016;HUB_04...,HUB_006:2;HUB_012:1;HUB_014:1;HUB_015:1;HUB_01...
8,5,infeasible,4.334708,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,6,infeasible,4.679531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 10. Primary endogenous infrastructure model

The fixed-$p$ experiments are useful diagnostics, but the main operational policy question is:

> What is the minimum infrastructure required to satisfy the operational requirements?

The primary formulation therefore allows the optimizer to choose the number of active hubs.

A three-stage lexicographic objective is used.

### Stage 1: minimum active hubs

$$
\min \sum_j y_j.
$$

### Stage 2: minimum fleet

After fixing the minimum hub count:

$$
\min \sum_j n_j.
$$

### Stage 3: minimum response time

After fixing both minimum infrastructure levels:

$$
\min
\sum_i\sum_j
T^{resp}_{ij}x_{ij}.
$$

This hierarchy prioritizes infrastructure parsimony first, fleet parsimony second, and service efficiency third without introducing arbitrary objective weights.

In [ ]:
def solve_endogenous(
    total_packages=30,
    safe_roundtrip_km=20.0,
    response_target_min=5.0,
    turnaround_min=30.0,
    redundancy=2
):

    q = allocate_integer_demand(
        total_packages
    )

    feasible = feasibility_matrix(
        safe_roundtrip_km,
        response_target_min
    )

    cycle = cycle_matrix(
        turnaround_min
    )

    active = np.where(q > 0)[0]

    if any(
        not feasible[i].any()
        for i in active
    ):
        return {
            "termination": "infeasible_reachability"
        }, None

    if redundancy > 1:

        bad = [
            i for i in critical_indices
            if feasible[i].sum() < redundancy
        ]

        if bad:
            return {
                "termination": "infeasible_redundancy_geometry",
                "critical_cells_without_backup": len(bad)
            }, None

    m = pyo.ConcreteModel()

    m.I = pyo.RangeSet(0, len(I) - 1)
    m.J = pyo.RangeSet(0, len(J) - 1)

    m.y = pyo.Var(
        m.J,
        domain=pyo.Binary
    )

    m.n = pyo.Var(
        m.J,
        domain=pyo.NonNegativeIntegers,
        bounds=(0, MAX_DRONES_PER_HUB)
    )

    m.x = pyo.Var(
        m.I,
        m.J,
        domain=pyo.NonNegativeIntegers
    )

    m.demand = pyo.Constraint(
        m.I,
        rule=lambda m, i:
        sum(m.x[i, j] for j in m.J)
        == int(q[i])
    )

    m.link = pyo.Constraint(
        m.I,
        m.J,
        rule=lambda m, i, j:
        m.x[i, j]
        <= int(q[i]) * m.y[j]
    )

    m.fleet_lower = pyo.Constraint(
        m.J,
        rule=lambda m, j:
        m.n[j] >= m.y[j]
    )

    m.fleet_upper = pyo.Constraint(
        m.J,
        rule=lambda m, j:
        m.n[j]
        <= MAX_DRONES_PER_HUB * m.y[j]
    )

    m.forbidden = pyo.ConstraintList()

    for i in I:
        for j in J:
            if not feasible[i, j]:
                m.forbidden.add(
                    m.x[i, j] == 0
                )

    m.capacity = pyo.Constraint(
        m.J,
        rule=lambda m, j:
        sum(
            float(cycle[i, j])
            * m.x[i, j]
            for i in I
        )
        <= 60 * m.n[j]
    )

    m.backup = pyo.ConstraintList()

    if redundancy > 1:

        for i in critical_indices:

            feasible_hubs = [
                j for j in J
                if feasible[i, j]
            ]

            m.backup.add(
                sum(
                    m.y[j]
                    for j in feasible_hubs
                )
                >= redundancy
            )

    total_solve_time = 0.0

    # --------------------------------------------------------
    # Stage 1: minimum hubs
    # --------------------------------------------------------

    m.objective_hubs = pyo.Objective(
        expr=sum(
            m.y[j]
            for j in m.J
        )
    )

    t = time.perf_counter()

    r1 = solver.solve(
        m,
        load_solutions=False
    )

    total_solve_time += (
        time.perf_counter() - t
    )

    term1 = str(
        r1.solver.termination_condition
    )

    if term1.lower() != "optimal":
        return {
            "termination": term1,
            "solve_time_s": total_solve_time
        }, None

    _load_solution(m, r1)

    minimum_hubs = int(
        round(
            sum(
                pyo.value(m.y[j])
                for j in J
            )
        )
    )

    # --------------------------------------------------------
    # Stage 2: minimum drones
    # --------------------------------------------------------

    m.hub_fix = pyo.Constraint(
        expr=sum(
            m.y[j]
            for j in m.J
        )
        == minimum_hubs
    )

    m.objective_hubs.deactivate()

    m.objective_drones = pyo.Objective(
        expr=sum(
            m.n[j]
            for j in m.J
        )
    )

    t = time.perf_counter()

    r2 = solver.solve(
        m,
        load_solutions=False
    )

    total_solve_time += (
        time.perf_counter() - t
    )

    term2 = str(
        r2.solver.termination_condition
    )

    if term2.lower() != "optimal":
        return {
            "termination": term2,
            "minimum_hubs": minimum_hubs,
            "solve_time_s": total_solve_time
        }, None

    _load_solution(m, r2)

    minimum_drones = int(
        round(
            sum(
                pyo.value(m.n[j])
                for j in J
            )
        )
    )

    # --------------------------------------------------------
    # Stage 3: minimum response time
    # --------------------------------------------------------

    m.fleet_fix = pyo.Constraint(
        expr=sum(
            m.n[j]
            for j in m.J
        )
        == minimum_drones
    )

    m.objective_drones.deactivate()

    m.objective_response = pyo.Objective(
        expr=sum(
            float(RESPONSE_MIN[i, j])
            * m.x[i, j]
            for i in I
            for j in J
        )
    )

    t = time.perf_counter()

    r3 = solver.solve(
        m,
        load_solutions=False
    )

    total_solve_time += (
        time.perf_counter() - t
    )

    term3 = str(
        r3.solver.termination_condition
    )

    if term3.lower() != "optimal":
        return {
            "termination": term3,
            "minimum_hubs": minimum_hubs,
            "minimum_drones": minimum_drones,
            "solve_time_s": total_solve_time
        }, None

    _load_solution(m, r3)

    selected = [
        j for j in J
        if pyo.value(m.y[j]) > 0.5
    ]

    drones = {
        j: int(round(pyo.value(m.n[j])))
        for j in selected
    }

    flows = {
        (i, j): int(round(pyo.value(m.x[i, j])))
        for i in I
        for j in J
        if (
            pyo.value(m.x[i, j]) is not None
            and pyo.value(m.x[i, j]) > 0.5
        )
    }

    responses = []
    distances = []

    for (i, j), units in flows.items():

        responses.extend(
            [RESPONSE_MIN[i, j]] * units
        )

        distances.extend(
            [D[i, j]] * units
        )

    metrics = {
        "packages_per_hour": total_packages,
        "safe_roundtrip_km": safe_roundtrip_km,
        "response_target_min": response_target_min,
        "turnaround_min": turnaround_min,
        "redundancy": redundancy,
        "minimum_hubs": minimum_hubs,
        "minimum_drones": minimum_drones,
        "mean_response_min": float(np.mean(responses)),
        "max_response_min": float(np.max(responses)),
        "mean_oneway_distance_km": float(np.mean(distances)),
        "max_oneway_distance_km": float(np.max(distances)),
        "termination": term3,
        "solve_time_s": total_solve_time,
        "selected_hubs": ";".join(
            hubs.iloc[selected]["hub_id"].astype(str)
        ),
        "drone_allocation": ";".join(
            f'{hubs.iloc[j]["hub_id"]}:{drones[j]}'
            for j in selected
        )
    }

    solution = {
        "selected": selected,
        "drones": drones,
        "flows": flows,
        "q": q,
        "feasible": feasible,
        "cycle": cycle
    }

    return metrics, solution

## 11. Baseline endogenous comparison

The endogenous model is solved under identical baseline assumptions with $r=1$ and
$r=2$, giving the infrastructure cost of resilience with the hub count determined by the
model rather than fixed externally.

In [ ]:
baseline_endogenous_results = []
baseline_endogenous_solutions = {}

for redundancy in [1, 2]:

    metrics, solution = solve_endogenous(
        total_packages=BASELINE_PACKAGES_PER_HOUR,
        safe_roundtrip_km=BASELINE_SAFE_ROUNDTRIP_KM,
        response_target_min=BASELINE_RESPONSE_TARGET_MIN,
        turnaround_min=BASELINE_TURNAROUND_MIN,
        redundancy=redundancy
    )

    baseline_endogenous_results.append(
        metrics
    )

    baseline_endogenous_solutions[
        redundancy
    ] = solution

    print(
        "redundancy =", redundancy,
        "|", metrics["termination"],
        "| hubs =", metrics.get("minimum_hubs"),
        "| drones =", metrics.get("minimum_drones")
    )

baseline_endogenous_df = pd.DataFrame(
    baseline_endogenous_results
)

display(baseline_endogenous_df)

redundancy = 1 | optimal | hubs = 7 | drones = 19
redundancy = 2 | optimal | hubs = 10 | drones = 21


,packages_per_hour,safe_roundtrip_km,response_target_min,turnaround_min,redundancy,minimum_hubs,minimum_drones,mean_response_min,max_response_min,mean_oneway_distance_km,max_oneway_distance_km,termination,solve_time_s,selected_hubs,drone_allocation
0,30,20.0,5.0,30.0,1,7,19,1.837398,3.028929,0.753658,1.826036,optimal,14.087199,HUB_006;HUB_014;HUB_016;HUB_040;HUB_041;HUB_04...,HUB_006:3;HUB_014:2;HUB_016:3;HUB_040:3;HUB_04...
1,30,20.0,5.0,30.0,2,10,21,2.081827,4.858365,0.973644,3.472529,optimal,9.775939,HUB_006;HUB_014;HUB_015;HUB_035;HUB_040;HUB_04...,HUB_006:3;HUB_014:2;HUB_015:3;HUB_035:1;HUB_04...


## 12. Final experimental design

The **primary sensitivity experiment uses the endogenous model**, so the number of hubs is an output rather than an externally specified input.

A one-factor-at-a-time design is used around the resilient baseline. This makes the operational effect of each assumption interpretable while avoiding a large Cartesian product of partly arbitrary scenario parameters.

### Baseline

- workload: 30 standardized packages/hour;
- safe round-trip mission distance: 20 km;
- response target: 5 min;
- turnaround: 30 min;
- critical-area redundancy: 2 independent hubs;
- maximum fleet at any active hub: 3 drones.

### Demand-breadth sensitivity

$$
Q\in\{15,30,45\}\text{ packages/hour}.
$$

This measures how required hubs and fleet size scale with workload.

### Battery-endurance sensitivity

$$
L^{safe}\in\{10,15,20\}\text{ km round trip}.
$$

This determines whether aircraft endurance is binding once the response-time constraint is also imposed.

### Response-time sensitivity

$$
T^{\max}\in\{5,7,10\}\text{ min}.
$$

This measures the infrastructure cost of stricter emergency-response requirements.

### Turnaround sensitivity

$$
T^{turn}\in\{15,30,45\}\text{ min}.
$$

This measures the effect of charging, battery handling, re-equipment, and dispatch readiness on fleet throughput.

### Resilience sensitivity

$$
r\in\{1,2\}.
$$

This quantifies the additional spatial and fleet infrastructure required for backup preparedness.

Duplicate baseline configurations are removed automatically.

The fixed-$p$ sweep from Sections 8-9 remains a separate diagnostic experiment and is not mixed into this final sensitivity design.

In [ ]:
scenario_rows = []

def add_sensitivity_family(
    family,
    parameter,
    values
):

    for value in values:

        row = {
            "family": family,
            "packages_per_hour": BASELINE_PACKAGES_PER_HOUR,
            "safe_roundtrip_km": BASELINE_SAFE_ROUNDTRIP_KM,
            "response_target_min": BASELINE_RESPONSE_TARGET_MIN,
            "turnaround_min": BASELINE_TURNAROUND_MIN,
            "redundancy": BASELINE_REDUNDANCY
        }

        row[parameter] = value

        scenario_rows.append(row)


add_sensitivity_family(
    "demand",
    "packages_per_hour",
    [15, 30, 45]
)

add_sensitivity_family(
    "battery",
    "safe_roundtrip_km",
    [10.0, 15.0, 20.0]
)

add_sensitivity_family(
    "response",
    "response_target_min",
    [5.0, 7.0, 10.0]
)

add_sensitivity_family(
    "turnaround",
    "turnaround_min",
    [15.0, 30.0, 45.0]
)

add_sensitivity_family(
    "redundancy",
    "redundancy",
    [1, 2]
)

sensitivity_scenarios = (
    pd.DataFrame(scenario_rows)
    .drop_duplicates(
        subset=[
            "packages_per_hour",
            "safe_roundtrip_km",
            "response_target_min",
            "turnaround_min",
            "redundancy"
        ]
    )
    .reset_index(drop=True)
)

sensitivity_scenarios["scenario_id"] = [
    f"S{i+1:02d}"
    for i in range(
        len(sensitivity_scenarios)
    )
]

display(sensitivity_scenarios)

print(
    "Unique final sensitivity scenarios:",
    len(sensitivity_scenarios)
)

,family,packages_per_hour,safe_roundtrip_km,response_target_min,turnaround_min,redundancy,scenario_id
0,demand,15,20.0,5.0,30.0,2,S01
1,demand,30,20.0,5.0,30.0,2,S02
2,demand,45,20.0,5.0,30.0,2,S03
3,battery,30,10.0,5.0,30.0,2,S04
4,battery,30,15.0,5.0,30.0,2,S05
5,response,30,20.0,7.0,30.0,2,S06
6,response,30,20.0,10.0,30.0,2,S07
7,turnaround,30,20.0,5.0,15.0,2,S08
8,turnaround,30,20.0,5.0,45.0,2,S09
9,redundancy,30,20.0,5.0,30.0,1,S10


Unique final sensitivity scenarios: 10


## 13. Run the final endogenous sensitivity experiment

Each scenario independently determines:

- the minimum number of active hubs;
- the minimum number of drones at that hub count;
- the fastest package allocation at those minimum infrastructure levels.

All scenarios are solved exactly with HiGHS through Pyomo.

In [ ]:
sensitivity_results = []
sensitivity_solutions = {}

for _, s in sensitivity_scenarios.iterrows():

    metrics, solution = solve_endogenous(
        total_packages=int(
            s["packages_per_hour"]
        ),
        safe_roundtrip_km=float(
            s["safe_roundtrip_km"]
        ),
        response_target_min=float(
            s["response_target_min"]
        ),
        turnaround_min=float(
            s["turnaround_min"]
        ),
        redundancy=int(
            s["redundancy"]
        )
    )

    metrics["scenario_id"] = (
        s["scenario_id"]
    )

    metrics["scenario_family"] = (
        s["family"]
    )

    sensitivity_results.append(
        metrics
    )

    sensitivity_solutions[
        s["scenario_id"]
    ] = solution

    print(
        s["scenario_id"],
        s["family"],
        "|", metrics["termination"],
        "| hubs =", metrics.get("minimum_hubs"),
        "| drones =", metrics.get("minimum_drones")
    )

sensitivity_results_df = pd.DataFrame(
    sensitivity_results
)

display(sensitivity_results_df)

S01 demand | optimal | hubs = 8 | drones = 13
S02 demand | optimal | hubs = 10 | drones = 21
S03 demand | optimal | hubs = 11 | drones = 29
S04 battery | optimal | hubs = 10 | drones = 21
S05 battery | optimal | hubs = 10 | drones = 21
S06 response | optimal | hubs = 8 | drones = 20
S07 response | optimal | hubs = 7 | drones = 19
S08 turnaround | optimal | hubs = 8 | drones = 12
S09 turnaround | optimal | hubs = 12 | drones = 32
S10 redundancy | optimal | hubs = 7 | drones = 19


,packages_per_hour,safe_roundtrip_km,response_target_min,turnaround_min,redundancy,minimum_hubs,minimum_drones,mean_response_min,max_response_min,mean_oneway_distance_km,max_oneway_distance_km,termination,solve_time_s,selected_hubs,drone_allocation,scenario_id,scenario_family
0,15,20.0,5.0,30.0,2,8,13,2.122672,3.782790,1.010405,2.504511,optimal,8.525014,HUB_014;HUB_015;HUB_036;HUB_040;HUB_042;HUB_04...,HUB_014:2;HUB_015:1;HUB_036:1;HUB_040:3;HUB_04...,S01,demand
1,30,20.0,5.0,30.0,2,10,21,2.081827,4.858365,0.973644,3.472529,optimal,9.983143,HUB_006;HUB_014;HUB_015;HUB_035;HUB_040;HUB_04...,HUB_006:3;HUB_014:2;HUB_015:3;HUB_035:1;HUB_04...,S02,demand
2,45,20.0,5.0,30.0,2,11,29,2.470198,4.831301,1.323178,3.448171,optimal,9.085329,HUB_006;HUB_014;HUB_015;HUB_031;HUB_036;HUB_04...,HUB_006:3;HUB_014:3;HUB_015:3;HUB_031:1;HUB_03...,S03,demand
3,30,10.0,5.0,30.0,2,10,21,2.081827,4.858365,0.973644,3.472529,optimal,9.628072,HUB_006;HUB_014;HUB_015;HUB_035;HUB_040;HUB_04...,HUB_006:3;HUB_014:2;HUB_015:3;HUB_035:1;HUB_04...,S04,battery
4,30,15.0,5.0,30.0,2,10,21,2.081827,4.858365,0.973644,3.472529,optimal,11.364291,HUB_006;HUB_014;HUB_015;HUB_035;HUB_040;HUB_04...,HUB_006:3;HUB_014:2;HUB_015:3;HUB_035:1;HUB_04...,S05,battery
5,30,20.0,7.0,30.0,2,8,20,2.032069,6.571024,0.928862,5.013922,optimal,9.304204,HUB_007;HUB_016;HUB_040;HUB_041;HUB_042;HUB_04...,HUB_007:1;HUB_016:3;HUB_040:3;HUB_041:3;HUB_04...,S06,response
6,30,20.0,10.0,30.0,2,7,19,2.493304,7.224773,1.343974,5.602296,optimal,11.944862,HUB_016;HUB_040;HUB_041;HUB_042;HUB_045;HUB_04...,HUB_016:3;HUB_040:3;HUB_041:3;HUB_042:3;HUB_04...,S07,response
7,30,20.0,5.0,15.0,2,8,12,2.162051,4.858365,1.045846,3.472529,optimal,8.146809,HUB_014;HUB_015;HUB_035;HUB_036;HUB_040;HUB_04...,HUB_014:1;HUB_015:1;HUB_035:1;HUB_036:1;HUB_04...,S08,turnaround
8,30,20.0,5.0,45.0,2,12,32,2.587161,4.949333,1.428445,3.554400,optimal,7.281136,HUB_006;HUB_012;HUB_014;HUB_015;HUB_036;HUB_03...,HUB_006:3;HUB_012:3;HUB_014:3;HUB_015:3;HUB_03...,S09,turnaround
9,30,20.0,5.0,30.0,1,7,19,1.837398,3.028929,0.753658,1.826036,optimal,13.397429,HUB_006;HUB_014;HUB_016;HUB_040;HUB_041;HUB_04...,HUB_006:3;HUB_014:2;HUB_016:3;HUB_040:3;HUB_04...,S10,redundancy


## 14. Save experiment checkpoints

Summary and detailed solution data are written to
`data/model/operational_resilient_solutions_v4/`, so downstream analysis can read the
results without re-solving.

In [ ]:
# ============================================================
# SECTION 14
# SAVE ALL EXPERIMENT CHECKPOINTS
# ============================================================

print("=" * 70)
print("SAVING NOTEBOOK 07 V4 RESULTS")
print("=" * 70)


# ------------------------------------------------------------
# 1. Reconstruct summary DataFrames when necessary
# ------------------------------------------------------------

# Fixed-p baseline sweep
if "fixed_p_baseline_df" not in globals():

    if "fixed_p_baseline_results" in globals():

        fixed_p_baseline_df = pd.DataFrame(
            fixed_p_baseline_results
        )

        print(
            "Reconstructed:",
            "fixed_p_baseline_df"
        )

    else:

        print(
            "WARNING:",
            "fixed-p baseline results are not in memory."
        )

        fixed_p_baseline_df = None


# Resilience fixed-p comparison
if "resilience_df" not in globals():

    if "resilience_results" in globals():

        resilience_df = pd.DataFrame(
            resilience_results
        )

        print(
            "Reconstructed:",
            "resilience_df"
        )

    else:

        print(
            "WARNING:",
            "resilience results are not in memory."
        )

        resilience_df = None


# Baseline endogenous comparison
if "baseline_endogenous_df" not in globals():

    if "baseline_endogenous_results" in globals():

        baseline_endogenous_df = pd.DataFrame(
            baseline_endogenous_results
        )

        print(
            "Reconstructed:",
            "baseline_endogenous_df"
        )

    else:

        print(
            "WARNING:",
            "baseline endogenous results are not in memory."
        )

        baseline_endogenous_df = None


# Final sensitivity results
if "sensitivity_results_df" not in globals():

    if "sensitivity_results" in globals():

        sensitivity_results_df = pd.DataFrame(
            sensitivity_results
        )

        print(
            "Reconstructed:",
            "sensitivity_results_df"
        )

    else:

        print(
            "WARNING:",
            "sensitivity results are not in memory."
        )

        sensitivity_results_df = None


# ------------------------------------------------------------
# 2. Ensure output directory exists
# ------------------------------------------------------------

SOLUTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "\nSaving to:"
)

print(
    SOLUTION_DIR.resolve()
)


# ------------------------------------------------------------
# 3. Save run-level tables
# ------------------------------------------------------------

saved_files = []


if fixed_p_baseline_df is not None:

    path = (
        SOLUTION_DIR
        / "fixed_p_baseline_sweep_v4.csv"
    )

    fixed_p_baseline_df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    saved_files.append(path)


if resilience_df is not None:

    path = (
        SOLUTION_DIR
        / "fixed_p_resilience_comparison_v4.csv"
    )

    resilience_df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    saved_files.append(path)


if baseline_endogenous_df is not None:

    path = (
        SOLUTION_DIR
        / "baseline_endogenous_comparison_v4.csv"
    )

    baseline_endogenous_df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    saved_files.append(path)


if "sensitivity_scenarios" in globals():

    path = (
        SOLUTION_DIR
        / "sensitivity_scenarios_v4.csv"
    )

    sensitivity_scenarios.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    saved_files.append(path)


if sensitivity_results_df is not None:

    path = (
        SOLUTION_DIR
        / "endogenous_sensitivity_results_v4.csv"
    )

    sensitivity_results_df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    saved_files.append(path)


# ------------------------------------------------------------
# 4. Save detailed sensitivity solutions
# ------------------------------------------------------------

hub_records = []
flow_records = []


if "sensitivity_solutions" in globals():

    for scenario_id, solution in (
        sensitivity_solutions.items()
    ):

        if solution is None:
            continue


        # ----------------------------------------------------
        # Selected hubs and drone fleet
        # ----------------------------------------------------

        for j in solution["selected"]:

            h = hubs.iloc[j]

            hub_records.append({

                "scenario_id":
                    scenario_id,

                "hub_id":
                    h["hub_id"],

                "hub_name":
                    h.get(
                        "map_name",
                        ""
                    ),

                "category":
                    h["category"],

                "hajj_zone":
                    h["hajj_zone"],

                "latitude":
                    h["latitude"],

                "longitude":
                    h["longitude"],

                "drones":
                    solution["drones"][j]
            })


        # ----------------------------------------------------
        # Package flows
        # ----------------------------------------------------

        for (i, j), units in (
            solution["flows"].items()
        ):

            d = demand.iloc[i]

            h = hubs.iloc[j]

            flow_records.append({

                "scenario_id":
                    scenario_id,

                "demand_point_id":
                    d["demand_point_id"],

                "demand_zone":
                    d["hajj_zone"],

                "hub_id":
                    h["hub_id"],

                "packages_per_hour":
                    units,

                "oneway_distance_km":
                    D[i, j],

                "roundtrip_distance_km":
                    ROUNDTRIP_KM[i, j],

                "response_min":
                    RESPONSE_MIN[i, j],

                "mission_cycle_min":
                    solution["cycle"][i, j]
            })


else:

    print(
        "\nWARNING:",
        "sensitivity_solutions is not in memory."
    )


# ------------------------------------------------------------
# 5. Convert detailed records to DataFrames
# ------------------------------------------------------------

selected_hubs_fleet_df = pd.DataFrame(
    hub_records
)

package_flows_df = pd.DataFrame(
    flow_records
)


# ------------------------------------------------------------
# 6. Save detailed records
# ------------------------------------------------------------

if not selected_hubs_fleet_df.empty:

    path = (
        SOLUTION_DIR
        / "selected_hubs_fleet_v4.csv"
    )

    selected_hubs_fleet_df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    saved_files.append(path)


if not package_flows_df.empty:

    path = (
        SOLUTION_DIR
        / "package_flows_v4.csv"
    )

    package_flows_df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    saved_files.append(path)


# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------

print(
    "\n" + "=" * 70
)

print(
    "FILES SAVED"
)

print(
    "=" * 70
)


for path in saved_files:

    size_kb = (
        path.stat().st_size
        / 1024
    )

    print(
        f"{path.name:<45}"
        f"{size_kb:>10.1f} KB"
    )


print(
    "\nSummary"
)

print(
    "-" * 70
)


if fixed_p_baseline_df is not None:

    print(
        "Fixed-p baseline runs:",
        len(fixed_p_baseline_df)
    )


if resilience_df is not None:

    print(
        "Fixed-p resilience runs:",
        len(resilience_df)
    )


if baseline_endogenous_df is not None:

    print(
        "Baseline endogenous runs:",
        len(baseline_endogenous_df)
    )


if sensitivity_results_df is not None:

    print(
        "Sensitivity runs:",
        len(sensitivity_results_df)
    )


print(
    "Selected hub/fleet records:",
    len(selected_hubs_fleet_df)
)

print(
    "Package-flow records:",
    len(package_flows_df)
)


print(
    "\nCheckpoint save complete."
)

SAVING NOTEBOOK 07 V4 RESULTS

Saving to:
C:\Users\firas.hindawi.CCM\Desktop\papers\Drone-Based Data Driven Medical Supply Distribution\hajj-drone-logistics\notebooks\hajj_drone_project\data\model\operational_resilient_solutions_v4

FILES SAVED
fixed_p_resilience_comparison_v4.csv                3.3 KB
baseline_endogenous_comparison_v4.csv               0.8 KB
sensitivity_scenarios_v4.csv                        0.4 KB
endogenous_sensitivity_results_v4.csv               3.2 KB
selected_hubs_fleet_v4.csv                          8.4 KB
package_flows_v4.csv                               34.0 KB

Summary
----------------------------------------------------------------------
Fixed-p resilience runs: 16
Baseline endogenous runs: 2
Sensitivity runs: 10
Selected hub/fleet records: 91
Package-flow records: 300

Checkpoint save complete.
